# NYC 311 Bronze Layer Ingestion

This notebook ingests raw NYC 311 complaint data from S3 into the bronze layer.

**Source**: S3 external location `db_s3_external_databricks-s3-ingest-civiclens` → `s3://civiclens-data/nyc311/`

**Target**: `civic_lens.bronze.nyc_311_raw`

**Schema**: 
- unique_key, created_date, closed_date, complaint_type, descriptor
- agency, borough, latitude, longitude, status, resolution_description

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
# Set up catalog and schema
catalog = "civic_lens"
schema = "bronze"

# Create schema if it doesn't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

print(f"Using: {catalog}.{schema}")

In [0]:
# External location path for NYC 311 data
external_location = "db_s3_external_databricks-s3-ingest-civiclens"
s3_path = f"{external_location}/nyc311/"

print(f"Reading from: {s3_path}")

In [0]:
# Read all CSV files from the NYC 311 folder
# Use the actual S3 path for the external location
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .option("multiLine", "true")
    .option("escape", '"')
    .load("s3://civiclens-data/nyc311/")
)

print(f"Records loaded: {df_raw.count():,}")
print(f"\nSchema:")
df_raw.printSchema()

In [0]:
# Display sample records
display(df_raw.limit(10))

In [0]:
# Add metadata columns for data lineage and processing
df_bronze = (
    df_raw
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.col("_metadata.file_path"))
)

print(f"Bronze DataFrame ready with {df_bronze.count():,} records")

In [0]:
# Write to bronze table (overwrite mode for initial load)
target_table = f"{catalog}.{schema}.nyc_311_raw"

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(target_table)
)

print(f"✓ Data written to {target_table}")

In [0]:
# Verify the table was created successfully
df_verify = spark.table(target_table)

print(f"Table: {target_table}")
print(f"Record count: {df_verify.count():,}")
print(f"\nColumns: {df_verify.columns}")
print(f"\nSample data:")
display(df_verify.limit(5))